# 🚦 Smart City Mobility Analytics: Exploratory Data Analysis
### *Deep-Dive Spatial Telemetry, Congestion Profiling & Weather Impact Analysis*

This notebook explores the validated **Delta Lake Gold & Silver Layers** to analyze urban mobility patterns across six key metropolitan zones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load Simulated Gold Layer Fact Telemetry

In [ ]:
# Synthetic Telemetry Generator matching Delta Fact Schema
np.random.seed(42)
n_samples = 2500

zones = ["CBD", "TECHPARK", "TRAINSTATION", "AIRPORT", "HARBOR", "SUBURB"]
roads = ["R100", "R200", "R300", "R400", "R500"]
weather = ["CLEAR", "RAIN", "FOG", "STORM"]
vehicles = ["SEDAN", "SUV", "TRUCK", "BUS", "MOTORCYCLE", "EV_TAXI"]

df = pd.DataFrame({
    "vehicle_id": [f"VEH-{i:05d}" for i in range(n_samples)],
    "city_zone": np.random.choice(zones, n_samples, p=[0.25, 0.22, 0.20, 0.15, 0.08, 0.10]),
    "road_id": np.random.choice(roads, n_samples, p=[0.30, 0.25, 0.20, 0.15, 0.10]),
    "speed_kmh": np.random.normal(55, 20, n_samples).clip(15, 140).astype(int),
    "congestion_level": np.random.choice([1, 2, 3, 4, 5], n_samples, p=[0.15, 0.25, 0.30, 0.20, 0.10]),
    "weather": np.random.choice(weather, n_samples, p=[0.60, 0.25, 0.10, 0.05]),
    "vehicle_type": np.random.choice(vehicles, n_samples),
    "hour": np.random.randint(0, 24, n_samples)
})

df["peak_flag"] = df["hour"].apply(lambda h: 1 if (8 <= h <= 11 or 17 <= h <= 20) else 0)
df.head()

## 2. Zone Congestion & Speed Distribution

In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(x="city_zone", y="speed_kmh", hue="peak_flag", data=df, palette="viridis")
plt.title("Vehicle Velocity Distribution Across Urban Zones (Peak vs. Off-Peak)", fontsize=14, fontweight="bold")
plt.xlabel("Urban Sector", fontsize=12)
plt.ylabel("Velocity (km/h)", fontsize=12)
plt.legend(title="Rush Hour", labels=["Off-Peak", "Peak"])
plt.show()

## 3. Weather Regime Impact on Traffic Flow

In [ ]:
weather_stats = df.groupby("weather").agg({
    "speed_kmh": ["mean", "std"],
    "congestion_level": "mean"
})
weather_stats.columns = ["Avg_Speed_KMH", "Speed_StdDev", "Avg_Congestion"]
print("=== Weather Impact Summary ===")
print(weather_stats.round(2))